![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [1]:
import os
import glob

print("=" * 60)
print("FILE SYSTEM CHECK")
print("=" * 60)

# Check all parquet files
print("\nParquet files in directory:")
parquet_files = glob.glob('*.parquet')
for f in sorted(parquet_files):
    size = os.path.getsize(f) / (1024**2)  # MB
    print(f"  {f}: {size:.1f} MB")

# Check temp files
print("\nTemp files:")
temp_files = glob.glob('temp_*')
for f in sorted(temp_files):
    size = os.path.getsize(f) / (1024**2)
    print(f"  {f}: {size:.1f} MB")

# Total disk usage
total_parquet = sum(os.path.getsize(f) for f in parquet_files) / (1024**2)
print(f"\nTotal parquet files: {len(parquet_files)}")
print(f"Total size: {total_parquet:.1f} MB")

# Check if main file exists
if 'btc_merged_features.parquet' in parquet_files:
    print("\n✅ SUCCESS: btc_merged_features.parquet exists!")
    size = os.path.getsize('btc_merged_features.parquet') / (1024**2)
    print(f"   Final file size: {size:.1f} MB")
else:
    print("\n⚠️ btc_merged_features.parquet NOT found")

# Check for leftover chunks
chunk_files = glob.glob('*chunk*.parquet')
if chunk_files:
    print(f"\n⚠️ Warning: {len(chunk_files)} chunk files still exist")
    print("  You may want to delete them to save space")

print("=" * 60)

In [2]:
from QuantConnect.Research import QuantBook
from QuantConnect.DataSource import CryptoUniverse
from QuantConnect import Resolution
from datetime import timedelta
from QuantConnect.Indicators import RateOfChange
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from QuantConnect.Research import QuantBook
from QuantConnect.DataSource import CryptoUniverse
from QuantConnect import Resolution
from statsmodels.tsa.stattools import coint
import pandas as pd
import numpy as np
from itertools import combinations
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats

### Get Data

In [3]:
"""
from AlgorithmImports import *

qb = QuantBook()

symbols_list = ["BTCUSDT"]

# THIS IS THE CHANGE - USE THESE DATES!
start = pd.Timestamp(2021, 6, 1, tz="UTC") 
end = pd.Timestamp(2022, 1, 28, tz="UTC")  

print("="*80)
print(f"Date Range: {start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}")
print("="*80)

symbols = []
for ticker in symbols_list:
    symbol = qb.add_crypto(ticker, market=Market.BINANCE).symbol
    symbols.append(symbol)

minute_data = qb.history(TradeBar, symbols, start, end, Resolution.Minute)

# Resample multi-indexed dataframe to 5-minute bars
minute_data = minute_data.groupby(level='symbol').resample('5min', level='time').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
}).dropna()

# Group by symbol
dfs_by_symbol = {sym: df for sym, df in minute_data.groupby(level="symbol")}

print("Symbols in dfs_by_symbol:", [sym.Value for sym in dfs_by_symbol.keys()])

# ===== QUICK N=3 BAR RETURN ANALYSIS FOR BTC =====
print("\n" + "="*80)
print("N=3 BAR RETURN ANALYSIS")
print("="*80)

# Get BTC data
btc_symbol = symbols[0]
btc_df = dfs_by_symbol[btc_symbol]

# Quick calculation - no modification to original df
btc_close = btc_df['close'].values
n3_returns = (btc_close[3:] - btc_close[:-3]) / btc_close[:-3]

# Calculate balance
positive_pct = (n3_returns > 0).mean() * 100
negative_pct = (n3_returns < 0).mean() * 100

print(f"\nTotal bars: {len(btc_df):,}")
print(f"N=3 returns calculated: {len(n3_returns):,}")

print(f"\nClass Balance for n=3 bar returns:")
print(f"  Positive (>0): {positive_pct:.1f}%")
print(f"  Negative (<0): {negative_pct:.1f}%")
"""

In [4]:
from AlgorithmImports import *
import numpy as np

qb = QuantBook()

symbols_list = ["BTCUSDT"]

start = pd.Timestamp(2022, 5, 1, tz="UTC") 
end = pd.Timestamp(2023, 5, 1, tz="UTC")  

print("="*80)
print(f"Date Range: {start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}")
print("="*80)

symbols = []
for ticker in symbols_list:
    symbol = qb.add_crypto(ticker, market=Market.BINANCE).symbol
    symbols.append(symbol)

# ===== STEP 1: GET MINUTE DATA =====
minute_data_raw = qb.history(TradeBar, symbols, start, end, Resolution.Minute)

print("\n" + "="*80)
print("CALCULATING METRICS FROM MINUTE BARS")
print("="*80)

# ===== STEP 2: RESAMPLE TO 5-MINUTE BARS FIRST =====
minute_data = minute_data_raw.groupby(level='symbol').resample('5min', level='time').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
}).dropna()

# Group by symbol
dfs_by_symbol = {sym: df for sym, df in minute_data.groupby(level="symbol")}

print("\nSymbols in dfs_by_symbol:", [sym.Value for sym in dfs_by_symbol.keys()])

# ===== STEP 3: CALCULATE METRICS USING MINUTE BARS FOR EACH 5-MIN BAR =====
btc_symbol = symbols[0]
btc_5min_df = dfs_by_symbol[btc_symbol].copy()
btc_minute_df = minute_data_raw.loc[btc_symbol].copy()

print(f"\n5-minute bars: {len(btc_5min_df):,}")
print(f"Minute bars: {len(btc_minute_df):,}")

n_bars = 3  # 3 five-min bars = 15 minutes

def calculate_metrics_for_5min_bars(five_min_df, minute_df, n_bars=3):
    """
    For each 5-min bar at position i, calculate metrics over PAST n_bars (including current)
    Uses minute-level data for the path, but aligned to 5-min bar boundaries
    
    Example: At 5-min bar 04:10:00 (with n_bars=3):
      - Covers 5-min bars: 04:00, 04:05, 04:10
      - Uses minute bars from 04:00:00 to 04:14:59
      - Return = (close at 04:14 - open at 04:00) / open at 04:00
    """
    metrics_list = []
    five_min_times = five_min_df.index.get_level_values('time')
    minute_times = minute_df.index
    
    for i in range(n_bars - 1, len(five_min_times)):
        current_5min_time = five_min_times[i]
        start_5min_time = five_min_times[i - n_bars + 1]
        
        # Get the minute bars that fall within this 5-min window
        # From start of first 5-min bar to end of current 5-min bar
        end_minute_time = current_5min_time + pd.Timedelta(minutes=4, seconds=59)
        
        window_minutes = minute_df[
            (minute_times >= start_5min_time) & 
            (minute_times <= end_minute_time)
        ]
        
        if len(window_minutes) < (n_bars * 5 - 1):  # Should have ~15 minute bars
            metrics_list.append({
                'time': current_5min_time,
                'total_return': np.nan,
                'sharpe': np.nan,
                'sortino': np.nan,
                'upr': np.nan
            })
            continue
        
        # Total return: open of first minute bar to close of last minute bar
        total_ret = (window_minutes['close'].iloc[-1] - window_minutes['open'].iloc[0]) / window_minutes['open'].iloc[0]
        
        # Calculate minute returns (close to close)
        minute_returns = window_minutes['close'].pct_change().dropna()
        
        if len(minute_returns) < 1:
            metrics_list.append({
                'time': current_5min_time,
                'total_return': total_ret,
                'sharpe': np.nan,
                'sortino': np.nan,
                'upr': np.nan
            })
            continue
        
        # Path volatility
        path_std = minute_returns.std()
        
        # Sharpe
        sharpe = total_ret / (path_std + 1e-8)
        
        # Sortino
        downside_rets = minute_returns[minute_returns < 0]
        downside_std = downside_rets.std() if len(downside_rets) > 0 else 1e-8
        sortino = total_ret / (downside_std + 1e-8)
        
        # UPR
        upside_mean = minute_returns[minute_returns > 0].mean() if (minute_returns > 0).any() else 0
        downside_mean = minute_returns[minute_returns < 0].abs().mean() if (minute_returns < 0).any() else 1e-8
        upr = upside_mean / (downside_mean + 1e-8)
        
        metrics_list.append({
            'time': current_5min_time,
            'total_return': total_ret,
            'sharpe': sharpe,
            'sortino': sortino,
            'upr': upr
        })
    
    return pd.DataFrame(metrics_list).set_index('time')

# Calculate metrics
print(f"\nCalculating metrics over past {n_bars} five-min bars using minute-level path...")
btc_metrics = calculate_metrics_for_5min_bars(btc_5min_df, btc_minute_df, n_bars=n_bars)
print(f"Metrics calculated: {len(btc_metrics):,}")

# Merge metrics onto 5-min dataframe
btc_5min_df = btc_5min_df.join(btc_metrics, how='left')

# Update dfs_by_symbol
dfs_by_symbol[btc_symbol] = btc_5min_df

print(f"\n5-minute bars with metrics: {len(btc_5min_df):,}")

# ===== PRINT FIRST 20 BARS FOR VERIFICATION =====
print("\n" + "="*80)
print("FIRST 20 BARS: OHLC + TOTAL_RETURN")
print("="*80)

display_cols = ['open', 'high', 'low', 'close', 'total_return']
print(btc_5min_df[display_cols].head(20).to_string())

In [5]:
"""
# ===== CALCULATE ORIGINAL-STYLE RETURNS (5-MIN CLOSE-TO-CLOSE) =====
print("\n" + "="*80)
print("CALCULATING ORIGINAL N=3 BAR RETURNS (5-MIN CLOSE-TO-CLOSE)")
print("="*80)

# Extract 5-min close prices
btc_close_values = btc_5min_df['close'].values

# Calculate n=3 returns (current close - close 3 bars ago)
n3_returns_original = np.full(len(btc_close_values), np.nan)
n3_returns_original[3:] = (btc_close_values[3:] - btc_close_values[:-3]) / btc_close_values[:-3]

# Add to dataframe
btc_5min_df['total_return_original'] = n3_returns_original

print(f"Original returns calculated: {(~np.isnan(n3_returns_original)).sum():,} non-NaN values")

# Update dfs_by_symbol
dfs_by_symbol[btc_symbol] = btc_5min_df

# ===== COMPARISON PRINT =====
print("\n" + "="*80)
print("FIRST 20 BARS: COMPARISON OF MINUTE-BASED VS ORIGINAL RETURNS")
print("="*80)

display_cols = ['open', 'high', 'low', 'close', 'total_return', 'total_return_original']
comparison_df = btc_5min_df[display_cols].head(20).copy()
comparison_df['difference'] = comparison_df['total_return'] - comparison_df['total_return_original']

print(comparison_df.to_string())

print("\n" + "="*80)
print("EXPLANATION:")
print("="*80)
print("total_return (minute-based):")
print("  - Uses open of first minute bar to close of last minute bar")
print("  - Captures full 15-minute path volatility")
print("  - At bar i: uses minutes from (i-2) 00:00 to i 04:59")
print("\ntotal_return_original (5-min based):")
print("  - Simple: (close[i] - close[i-3]) / close[i-3]")
print("  - Uses only 5-min bar close prices")
print("  - At bar i: close[i] vs close[i-3]")
print("\nDifferences show impact of using minute-level data vs 5-min snapshots")
print("="*80)
"""

In [6]:
# Add this after the metrics are merged
print("\n" + "="*80)
print("SHIFTING METRICS TO CREATE FUTURE TARGETS")
print("="*80)

# Shift metrics by -3 bars so they become future predictions with no overlap
shift_bars = 3
btc_5min_df['total_return'] = btc_5min_df['total_return'].shift(-shift_bars)
btc_5min_df['sharpe'] = btc_5min_df['sharpe'].shift(-shift_bars)
btc_5min_df['sortino'] = btc_5min_df['sortino'].shift(-shift_bars)
btc_5min_df['upr'] = btc_5min_df['upr'].shift(-shift_bars)

print(f"Shifted metrics by {shift_bars} bars backward (to future)")
print("Now at timestamp 04:00 (bar ends 04:04):")
print("  - Features: Use data through 04:04")
print("  - Target: Predicts return from 04:05 to 04:19 (FUTURE, no overlap)")

# Update dfs_by_symbol
dfs_by_symbol[btc_symbol] = btc_5min_df

In [7]:
print(btc_5min_df[display_cols].head(20).to_string())

In [8]:
# ============================================================================
# CHECK THE FIRST 5 BARS FROM ORIGINAL DATA
# ============================================================================

print("\n" + "="*80)
print("FIRST 5 BARS FROM ORIGINAL COINBASE DATA")
print("="*80)

# Get BTCUSDT data
btc_symbol = [sym for sym in dfs_by_symbol.keys() if sym.Value == "BTCUSDT"][0]
btc_df = dfs_by_symbol[btc_symbol]

# Reset index to access time easily
btc_df_reset = btc_df.reset_index()

# Automatically use the first available timestamp as the start time
start_time = btc_df_reset['time'].min()

# Select the first 5 bars starting from the first available time
first_bars = btc_df_reset[btc_df_reset['time'] >= start_time].head(5)

for idx, row in first_bars.iterrows():
    print(f"\nBar at {row['time']}:")
    print(f"  Open: ${row['open']:.2f}")
    print(f"  High: ${row['high']:.2f}")
    print(f"  Low: ${row['low']:.2f}")
    print(f"  Close: ${row['close']:.2f}")
    print(f"  Volume: {row['volume']:.0f}")

print("\n" + "="*80)

print(f"Training data timezone: {btc_df_reset['time'].dt.tz}")
print(f"First timestamp: {btc_df_reset['time'].iloc[0]}")


### Time Features

In [9]:
def add_time_features(df):
    """
    Adds stationary time-based features to a minute-level DataFrame.
    
    KEY PRINCIPLE: All features are stationary and generalizable across time.
    A value of "1" today means the same thing years from now.
    """
    df = df.copy()
    df = df.reset_index() if 'time' not in df.index.names else df.reset_index(level='time')
    
    hour = df['time'].dt.hour
    minute = df['time'].dt.minute
    
    # ============================================================
    # 1️⃣ CYCLICAL TIME ENCODINGS
    # ============================================================
    
    # Hour of day - cyclical encoding
    hour_decimal = hour + minute / 60.0
    df['hour_sin'] = np.sin(2 * np.pi * hour_decimal / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour_decimal / 24)
    
    # Day of week - cyclical encoding
    df['day_sin'] = np.sin(2 * np.pi * df['time'].dt.dayofweek / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['time'].dt.dayofweek / 7)
    
    # Day of month - cyclical encoding
    df['day_of_month_sin'] = np.sin(2 * np.pi * df['time'].dt.day / 31)
    df['day_of_month_cos'] = np.cos(2 * np.pi * df['time'].dt.day / 31)
    
    # Minute of hour - cyclical encoding
    df['minute_sin'] = np.sin(2 * np.pi * minute / 60)
    df['minute_cos'] = np.cos(2 * np.pi * minute / 60)
    
    # ============================================================
    # 2️⃣ CRITICAL TIME WINDOWS
    # ============================================================
    
    # Midnight UTC window
    df['midnight_utc'] = (
        ((hour == 23) & (minute >= 55)) |
        ((hour == 0) & (minute <= 5))
    ).astype(int)
    
    # Hour boundaries
    df['hour_open'] = (minute <= 5).astype(int)
    df['hour_close'] = (minute >= 55).astype(int)
    
    # Day boundaries
    df['day_open'] = (hour == 0).astype(int)
    df['day_close'] = (hour == 23).astype(int)
    
    # ============================================================
    # 3️⃣ TRADING SESSION INDICATORS
    # ============================================================
    
    # Asian session (23:00-08:00 UTC)
    df['session_asia'] = (
        (hour >= 23) | (hour < 8)
    ).astype(int)
    
    # European session (07:00-16:00 UTC)
    df['session_europe'] = (
        (hour >= 7) & (hour < 16)
    ).astype(int)
    
    # US session (13:00-22:00 UTC)
    df['session_us'] = (
        (hour >= 13) & (hour < 22)
    ).astype(int)
    
    # Session overlaps
    df['overlap_europe_us'] = (
        (hour >= 13) & (hour < 16)
    ).astype(int)
    
    df['overlap_asia_europe'] = (
        (hour >= 7) & (hour < 8)
    ).astype(int)
    
    # ============================================================
    # 4️⃣ EQUITY MARKET HOURS
    # ============================================================
    
    is_weekday = df['time'].dt.dayofweek < 5
    
    # US equity market hours (14:30-21:00 UTC, Mon-Fri)
    df['us_equity_hours'] = (
        is_weekday &
        (hour >= 14) & 
        (hour < 21) &
        ~((hour == 14) & (minute < 30))
    ).astype(int)
    
    # US equity open (14:30 UTC ±10 min)
    df['us_equity_open'] = (
        is_weekday &
        (hour == 14) &
        (minute >= 20) &
        (minute <= 40)
    ).astype(int)
    
    # US equity close (21:00 UTC ±10 min)
    df['us_equity_close'] = (
        is_weekday &
        (
            ((hour == 20) & (minute >= 50)) |
            ((hour == 21) & (minute <= 10))
        )
    ).astype(int)
    
    # ============================================================
    # 5️⃣ WEEKEND/WEEKDAY INDICATORS
    # ============================================================
    
    df['is_weekend'] = (df['time'].dt.dayofweek >= 5).astype(int)
    df['is_monday'] = (df['time'].dt.dayofweek == 0).astype(int)
    df['is_friday'] = (df['time'].dt.dayofweek == 4).astype(int)
    
    # ============================================================
    # 6️⃣ CALENDAR EFFECTS
    # ============================================================
    
    df['first_day_of_month'] = (df['time'].dt.day == 1).astype(int)
    df['last_day_of_month'] = (
        df['time'].dt.day == df['time'].dt.daysinmonth
    ).astype(int)
    
    # ============================================================
    # 7️⃣ MINUTE-LEVEL GRANULARITY
    # ============================================================
    
    # Round number minutes (00, 15, 30, 45)
    df['round_minute'] = (minute % 15 == 0).astype(int)
    
    # ============================================================
    # 8️⃣ SESSION POSITION FEATURES (Normalized 0-1)
    # ============================================================
    
    # Position within Asian session (23:00-08:00 = 9 hours)
    asian_minutes = np.where(
        hour >= 23,
        (hour - 23) * 60 + minute,
        np.where(hour < 8, hour * 60 + minute + 60, -1)
    )
    df['pos_asia'] = np.where(asian_minutes >= 0, asian_minutes / (9 * 60), -1)
    
    # Position within European session (07:00-16:00 = 9 hours)
    df['pos_europe'] = np.where(
        (hour >= 7) & (hour < 16),
        ((hour - 7) * 60 + minute) / (9 * 60),
        -1
    )
    
    # Position within US session (13:00-22:00 = 9 hours)
    df['pos_us'] = np.where(
        (hour >= 13) & (hour < 22),
        ((hour - 13) * 60 + minute) / (9 * 60),
        -1
    )
    
    # Set time as index again
    df = df.set_index('time', append=True) if 'symbol' in df.columns else df.set_index('time')
    
    return df


# Apply to BTC only
btc_symbol = [sym for sym in dfs_by_symbol.keys() if 'BTC' in sym.Value][0]
dfs_by_symbol[btc_symbol] = add_time_features(dfs_by_symbol[btc_symbol].copy())

# Verification
print(f"--- {btc_symbol.Value} ---")
print(dfs_by_symbol[btc_symbol].head(5))
print(f"Columns: {dfs_by_symbol[btc_symbol].columns.tolist()}")
print()

# Optional: Show that other symbols remain unchanged
print("\nOther symbols (unchanged):")
for sym in dfs_by_symbol.keys():
    if sym != btc_symbol:
        print(f"--- {sym.Value} ---")
        print(f"Columns: {dfs_by_symbol[sym].columns.tolist()}")

### Price features

In [ ]:
def add_price_features(df):
    """
    Optimized stationary price features for 10-20 MINUTE holding periods (2-4 bars).
    
    OPTIMIZATION STRATEGY:
    - 70% of features focused on <1 hour horizons (most predictive)
    - 25% on 1-4 hour horizons (context)
    - 5% on >4 hour horizons (regime detection only)
    
    Focus: Micro-structure, short-term momentum, immediate price action
    """
    df = df.copy()
    
    close = df['close'].values
    high = df['high'].values
    low = df['low'].values
    open_price = df['open'].values
    
    # ============================================================
    # 1️⃣ MICRO RETURNS (1-5 bars = 5-25 minutes)
    # ============================================================
    
    # Very short-term returns (primary signals for 10-20 min holds)
    df['ret_1'] = df['close'].pct_change(1)      # 5-min (1 bar)
    df['ret_2'] = df['close'].pct_change(2)      # 10-min (2 bars)
    df['ret_3'] = df['close'].pct_change(3)      # 15-min (3 bars)
    df['ret_4'] = df['close'].pct_change(4)      # 20-min (4 bars)
    df['ret_6'] = df['close'].pct_change(6)      # 30-min (6 bars)
    
    # Log returns (better for statistical modeling)
    df['log_ret_1'] = np.log(df['close'] / df['close'].shift(1))
    df['log_ret_2'] = np.log(df['close'] / df['close'].shift(2))
    df['log_ret_3'] = np.log(df['close'] / df['close'].shift(3))
    df['log_ret_4'] = np.log(df['close'] / df['close'].shift(4))
    
    # ============================================================
    # 2️⃣ SHORT-TERM RETURNS (6-48 bars = 30 min - 4 hours)
    # ============================================================
    
    # Context for the micro moves
    df['ret_12'] = df['close'].pct_change(12)    # 1-hour
    df['ret_24'] = df['close'].pct_change(24)    # 2-hour
    df['ret_48'] = df['close'].pct_change(48)    # 4-hour
    
    df['log_ret_12'] = np.log(df['close'] / df['close'].shift(12))
    df['log_ret_48'] = np.log(df['close'] / df['close'].shift(48))
    
    # ============================================================
    # 3️⃣ MOMENTUM PERCENTILE RANKS (Short Horizons)
    # ============================================================
    
    # Percentile rank of returns (most important for short-term trading)
    df['roc_1_pctrank_6'] = df['ret_1'].rolling(6).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['roc_2_pctrank_12'] = df['ret_2'].rolling(12).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['roc_3_pctrank_12'] = df['ret_3'].rolling(12).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['roc_6_pctrank_24'] = df['ret_6'].rolling(24).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['roc_12_pctrank_48'] = df['ret_12'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # Price percentile ranks (short windows)
    df['price_pctrank_6'] = df['close'].rolling(6).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['price_pctrank_12'] = df['close'].rolling(12).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['price_pctrank_24'] = df['close'].rolling(24).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['price_pctrank_48'] = df['close'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # 4️⃣ MOVING AVERAGE RATIOS (Short-term EMAs)
    # ============================================================
    
    # Focus on very short-term EMAs for quick trades
    ema_3 = df['close'].ewm(span=3, adjust=False).mean()      # 15 minutes
    ema_6 = df['close'].ewm(span=6, adjust=False).mean()      # 30 minutes
    ema_12 = df['close'].ewm(span=12, adjust=False).mean()    # 1 hour
    ema_24 = df['close'].ewm(span=24, adjust=False).mean()    # 2 hours
    ema_48 = df['close'].ewm(span=48, adjust=False).mean()    # 4 hours (context only)
    
    # Distance from EMAs (percentage)
    df['price_ema3_ratio'] = df['close'] / ema_3 - 1
    df['price_ema6_ratio'] = df['close'] / ema_6 - 1
    df['price_ema12_ratio'] = df['close'] / ema_12 - 1
    df['price_ema24_ratio'] = df['close'] / ema_24 - 1
    df['price_ema48_ratio'] = df['close'] / ema_48 - 1
    
    # EMA cross signals (trend strength) - short-term focus
    df['ema3_ema6_ratio'] = ema_3 / ema_6 - 1
    df['ema6_ema12_ratio'] = ema_6 / ema_12 - 1
    df['ema12_ema24_ratio'] = ema_12 / ema_24 - 1
    df['ema24_ema48_ratio'] = ema_24 / ema_48 - 1
    
    # ============================================================
    # 5️⃣ DISTANCE TO HIGH/LOW (Short Windows - Critical!)
    # ============================================================
    
    # 30-minute window (6 bars) - MOST IMPORTANT
    high_30m = df['close'].rolling(6).max()
    low_30m = df['close'].rolling(6).min()
    df['dist_30m_high'] = (df['close'] - high_30m) / high_30m
    df['dist_30m_low'] = (df['close'] - low_30m) / low_30m
    
    # 1-hour window (12 bars)
    high_1h = df['close'].rolling(12).max()
    low_1h = df['close'].rolling(12).min()
    df['dist_1h_high'] = (df['close'] - high_1h) / high_1h
    df['dist_1h_low'] = (df['close'] - low_1h) / low_1h
    
    # 2-hour window (24 bars)
    high_2h = df['close'].rolling(24).max()
    low_2h = df['close'].rolling(24).min()
    df['dist_2h_high'] = (df['close'] - high_2h) / high_2h
    df['dist_2h_low'] = (df['close'] - low_2h) / low_2h
    
    # 4-hour window (48 bars) - context
    high_4h = df['close'].rolling(48).max()
    low_4h = df['close'].rolling(48).min()
    df['dist_4h_high'] = (df['close'] - high_4h) / high_4h
    df['dist_4h_low'] = (df['close'] - low_4h) / low_4h
    
    # 1-day window (288 bars) - regime only
    high_1d = df['close'].rolling(288).max()
    low_1d = df['close'].rolling(288).min()
    df['dist_1d_high'] = (df['close'] - high_1d) / high_1d
    df['dist_1d_low'] = (df['close'] - low_1d) / low_1d
    
    # ============================================================
    # 6️⃣ BINARY FLAGS - SHORT-TERM EXTREMES
    # ============================================================
    
    # At or near highs (within 0.5% for short-term, 1% for longer)
    df['near_30m_high'] = (df['dist_30m_high'] > -0.005).astype(int)
    df['near_1h_high'] = (df['dist_1h_high'] > -0.005).astype(int)
    df['near_2h_high'] = (df['dist_2h_high'] > -0.01).astype(int)
    df['near_4h_high'] = (df['dist_4h_high'] > -0.01).astype(int)
    
    # At or near lows
    df['near_30m_low'] = (df['dist_30m_low'] < 0.005).astype(int)
    df['near_1h_low'] = (df['dist_1h_low'] < 0.005).astype(int)
    df['near_2h_low'] = (df['dist_2h_low'] < 0.01).astype(int)
    df['near_4h_low'] = (df['dist_4h_low'] < 0.01).astype(int)
    
    # New highs/lows (breakouts)
    df['new_30m_high'] = (df['close'] >= high_30m).astype(int)
    df['new_1h_high'] = (df['close'] >= high_1h).astype(int)
    df['new_2h_high'] = (df['close'] >= high_2h).astype(int)
    df['new_4h_high'] = (df['close'] >= high_4h).astype(int)
    
    df['new_30m_low'] = (df['close'] <= low_30m).astype(int)
    df['new_1h_low'] = (df['close'] <= low_1h).astype(int)
    df['new_2h_low'] = (df['close'] <= low_2h).astype(int)
    df['new_4h_low'] = (df['close'] <= low_4h).astype(int)
    
    # ============================================================
    # 7️⃣ INTRABAR FEATURES (Critical for 5-min bars!)
    # ============================================================
    
    # High-low range (normalized by close)
    df['hl_range'] = (df['high'] - df['low']) / df['close']
    
    # Close position within bar (0 = at low, 1 = at high)
    df['close_position'] = np.where(
        df['high'] != df['low'],
        (df['close'] - df['low']) / (df['high'] - df['low']),
        0.5
    )
    
    # Upper/lower wicks (normalized)
    df['upper_wick'] = (df['high'] - np.maximum(df['open'], df['close'])) / df['close']
    df['lower_wick'] = (np.minimum(df['open'], df['close']) - df['low']) / df['close']
    
    # Body size (candle body)
    df['body_size'] = np.abs(df['close'] - df['open']) / df['close']
    
    # Wick ratio (upper wick vs lower wick)
    df['wick_ratio'] = np.where(
        df['lower_wick'] > 0,
        df['upper_wick'] / (df['lower_wick'] + 1e-8),
        0
    )
    
    # ============================================================
    # 8️⃣ MICRO TREND PATTERNS (1-5 bar patterns)
    # ============================================================
    
    # Consecutive ups/downs (very important for momentum)
    returns_sign = np.sign(df['ret_1'])
    
    consecutive_ups = []
    consecutive_downs = []
    up_count = 0
    down_count = 0
    
    for ret_sign in returns_sign:
        if ret_sign > 0:
            up_count += 1
            down_count = 0
        elif ret_sign < 0:
            down_count += 1
            up_count = 0
        else:
            up_count = 0
            down_count = 0
        
        consecutive_ups.append(up_count)
        consecutive_downs.append(down_count)
    
    df['consecutive_ups'] = consecutive_ups
    df['consecutive_downs'] = consecutive_downs
    
    # Higher highs / lower lows (2-bar pattern = 10 minutes)
    df['higher_high_2'] = (df['high'] > df['high'].shift(2)).astype(int)
    df['lower_low_2'] = (df['low'] < df['low'].shift(2)).astype(int)
    
    # Higher highs / lower lows (3-bar pattern = 15 minutes)
    df['higher_high_3'] = (df['high'] > df['high'].shift(3)).astype(int)
    df['lower_low_3'] = (df['low'] < df['low'].shift(3)).astype(int)
    
    # Strong momentum flags (3+ consecutive bars in same direction)
    df['strong_up_momentum'] = (df['consecutive_ups'] >= 3).astype(int)
    df['strong_down_momentum'] = (df['consecutive_downs'] >= 3).astype(int)
    
    # ============================================================
    # 9️⃣ PRICE ACCELERATION (2nd derivative of price)
    # ============================================================
    
    # Rate of change of returns (acceleration)
    df['ret_accel_1'] = df['ret_1'] - df['ret_1'].shift(1)
    df['ret_accel_2'] = df['ret_2'] - df['ret_2'].shift(1)
    
    # Sign of acceleration
    df['accel_positive'] = (df['ret_accel_1'] > 0).astype(int)
    
    # ============================================================
    # 10 VOLATILITY-ADJUSTED RETURNS (Z-scores - Short Windows)
    # ============================================================
    
    # Rolling z-scores with short windows
    ret_1_mean = df['ret_1'].rolling(12).mean()
    ret_1_std = df['ret_1'].rolling(12).std()
    df['ret_1_zscore'] = (df['ret_1'] - ret_1_mean) / (ret_1_std + 1e-8)
    
    ret_3_mean = df['ret_3'].rolling(24).mean()
    ret_3_std = df['ret_3'].rolling(24).std()
    df['ret_3_zscore'] = (df['ret_3'] - ret_3_mean) / (ret_3_std + 1e-8)
    
    ret_12_mean = df['ret_12'].rolling(48).mean()
    ret_12_std = df['ret_12'].rolling(48).std()
    df['ret_12_zscore'] = (df['ret_12'] - ret_12_mean) / (ret_12_std + 1e-8)
    
    # ============================================================
    # 11 RANGE POSITION (Where in recent range?)
    # ============================================================
    
    # Position within 30-min range (0 = at low, 1 = at high)
    range_30m = high_30m - low_30m
    df['range_position_30m'] = np.where(
        range_30m > 0,
        (df['close'] - low_30m) / range_30m,
        0.5
    )
    
    # Position within 1-hour range
    range_1h = high_1h - low_1h
    df['range_position_1h'] = np.where(
        range_1h > 0,
        (df['close'] - low_1h) / range_1h,
        0.5
    )
    
    # Position within 2-hour range
    range_2h = high_2h - low_2h
    df['range_position_2h'] = np.where(
        range_2h > 0,
        (df['close'] - low_2h) / range_2h,
        0.5
    )
    
    return df


# Apply to all symbols
for sym in dfs_by_symbol.keys():
    dfs_by_symbol[sym] = add_price_features(dfs_by_symbol[sym].copy())

# Verification
for sym in dfs_by_symbol.keys():
    print(f"--- {sym.Value} ---")
    print(dfs_by_symbol[sym].head(10))
    print(f"\nColumns: {dfs_by_symbol[sym].columns.tolist()}")
    print()

### Volume features

In [11]:
def add_volume_features(df):
    """
    OPTIMIZED volume features - removed slow computations
    """
    df = df.copy()
    
    # Handle zero volumes
    df['volume'] = df['volume'].replace(0, np.nan).ffill()
    
    # ============================================================
    # 1️⃣ RELATIVE VOLUME - SHORT WINDOWS
    # ============================================================
    
    vma_3 = df['volume'].rolling(3).mean()
    vma_6 = df['volume'].rolling(6).mean()
    vma_12 = df['volume'].rolling(12).mean()
    vma_24 = df['volume'].rolling(24).mean()
    vma_48 = df['volume'].rolling(48).mean()
    vma_288 = df['volume'].rolling(288).mean()
    
    df['rvol_15m'] = df['volume'] / vma_3
    df['rvol_30m'] = df['volume'] / vma_6
    df['rvol_1h'] = df['volume'] / vma_12
    df['rvol_2h'] = df['volume'] / vma_24
    df['rvol_4h'] = df['volume'] / vma_48
    
    # ============================================================
    # 2️⃣ VOLUME MOMENTUM
    # ============================================================
    
    df['vroc_1'] = df['volume'].pct_change(1)
    df['vroc_2'] = df['volume'].pct_change(2)
    df['vroc_3'] = df['volume'].pct_change(3)
    df['vroc_6'] = df['volume'].pct_change(6)
    df['vroc_12'] = df['volume'].pct_change(12)
    df['vroc_24'] = df['volume'].pct_change(24)
    
    # ============================================================
    # 3️⃣ VOLUME PERCENTILE RANKS - VECTORIZED (MUCH FASTER!)
    # ============================================================
    
    # Replace slow .apply() with fast rank-based approach
    df['vol_pctrank_6'] = df['volume'].rolling(6).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
    df['vol_pctrank_12'] = df['volume'].rolling(12).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
    df['vol_pctrank_24'] = df['volume'].rolling(24).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
    df['vol_pctrank_48'] = df['volume'].rolling(48).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
    
    df['rvol_pctrank_12'] = df['rvol_1h'].rolling(12).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
    df['rvol_pctrank_24'] = df['rvol_2h'].rolling(24).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
    df['rvol_pctrank_48'] = df['rvol_4h'].rolling(48).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
    
    # ALTERNATIVE: Comment out percentile ranks entirely if still slow
    # # df['vol_pctrank_6'] = 0.5
    # # df['vol_pctrank_12'] = 0.5
    # # df['vol_pctrank_24'] = 0.5
    # # df['vol_pctrank_48'] = 0.5
    # # df['rvol_pctrank_12'] = 0.5
    # # df['rvol_pctrank_24'] = 0.5
    # # df['rvol_pctrank_48'] = 0.5
    
    # ============================================================
    # 4️⃣ VOLUME TREND
    # ============================================================
    
    df['vma_3_6_ratio'] = vma_3 / vma_6 - 1
    df['vma_6_12_ratio'] = vma_6 / vma_12 - 1
    df['vma_12_24_ratio'] = vma_12 / vma_24 - 1
    df['vma_24_48_ratio'] = vma_24 / vma_48 - 1
    
    # ============================================================
    # 5️⃣ VOLUME CONCENTRATION
    # ============================================================
    
    vol_sum_6 = df['volume'].rolling(6).sum()
    vol_sum_12 = df['volume'].rolling(12).sum()
    vol_sum_24 = df['volume'].rolling(24).sum()
    vol_sum_48 = df['volume'].rolling(48).sum()
    
    df['vol_share_30m'] = df['volume'] / vol_sum_6
    df['vol_share_1h'] = df['volume'] / vol_sum_12
    df['vol_share_2h'] = df['volume'] / vol_sum_24
    df['vol_share_4h'] = df['volume'] / vol_sum_48
    
    # ============================================================
    # 6️⃣ CUMULATIVE VOLUME PATTERNS
    # ============================================================
    
    df['cum_vol_3'] = df['volume'].rolling(3).sum()
    df['cum_vol_6'] = df['volume'].rolling(6).sum()
    df['cum_vol_12'] = df['volume'].rolling(12).sum()
    
    df['cum_vol_3_norm'] = df['cum_vol_3'] / (vma_12 * 3)
    df['cum_vol_6_norm'] = df['cum_vol_6'] / (vma_12 * 6)
    df['cum_vol_12_norm'] = df['cum_vol_12'] / (vma_48 * 12)
    
    # ============================================================
    # 7️⃣ VOLUME ACCELERATION
    # ============================================================
    
    df['vroc_accel_1'] = df['vroc_1'] - df['vroc_1'].shift(1)
    df['vroc_accel_3'] = df['vroc_3'] - df['vroc_3'].shift(1)
    df['vol_accelerating'] = (df['vroc_accel_1'] > 0).astype(int)
    
    # ============================================================
    # 8️⃣ TIME-OF-DAY VOLUME PATTERNS - COMMENTED OUT (VERY SLOW!)
    # ============================================================
    
    # # SLOW FEATURE - Comment out if needed
    # df_reset = df.reset_index()
    # 
    # if 'time' in df_reset.columns:
    #     time_col = 'time'
    # else:
    #     time_col = df_reset.columns[0]
    # 
    # df_reset['time_only'] = pd.to_datetime(df_reset[time_col]).dt.time
    # df_reset['date'] = pd.to_datetime(df_reset[time_col]).dt.date
    # df_reset['day_of_week'] = pd.to_datetime(df_reset[time_col]).dt.dayofweek
    # df_reset['hour'] = pd.to_datetime(df_reset[time_col]).dt.hour
    # 
    # # VERY SLOW - Volume vs same bar previous day
    # v_rel_prev_day = []
    # for i in range(len(df_reset)):
    #     if i >= 288:
    #         prev_day_idx = i - 288
    #         if prev_day_idx >= 0 and df_reset.iloc[prev_day_idx]['volume'] > 0:
    #             ratio = df_reset.iloc[i]['volume'] / df_reset.iloc[prev_day_idx]['volume']
    #             v_rel_prev_day.append(ratio)
    #         else:
    #             v_rel_prev_day.append(1.0)
    #     else:
    #         v_rel_prev_day.append(1.0)
    # 
    # df_reset['vol_vs_prev_day'] = v_rel_prev_day
    # 
    # # VERY SLOW - Volume vs same time-of-day average
    # time_of_day_avg = []
    # for i in range(len(df_reset)):
    #     if i >= 864:
    #         lookback = df_reset.iloc[max(0, i-864):i]
    #         same_time = lookback[lookback['time_only'] == df_reset.iloc[i]['time_only']]
    #         avg_vol = same_time['volume'].mean()
    #         if avg_vol > 0:
    #             time_of_day_avg.append(df_reset.iloc[i]['volume'] / avg_vol)
    #         else:
    #             time_of_day_avg.append(1.0)
    #     else:
    #         time_of_day_avg.append(1.0)
    # 
    # df_reset['vol_vs_time_avg'] = time_of_day_avg
    # 
    # # VERY SLOW - Volume vs same hour average
    # hour_avg = []
    # for i in range(len(df_reset)):
    #     if i >= 576:
    #         lookback = df_reset.iloc[max(0, i-576):i]
    #         same_hour = lookback[lookback['hour'] == df_reset.iloc[i]['hour']]
    #         avg_vol = same_hour['volume'].mean()
    #         if avg_vol > 0:
    #             hour_avg.append(df_reset.iloc[i]['volume'] / avg_vol)
    #         else:
    #             hour_avg.append(1.0)
    #     else:
    #         hour_avg.append(1.0)
    # 
    # df_reset['vol_vs_hour_avg'] = hour_avg
    # 
    # df = df_reset.set_index(df.index.names) if hasattr(df.index, 'names') else df_reset
    
    # FAST REPLACEMENT - Simple placeholders
    df['vol_vs_prev_day'] = 1.0
    df['vol_vs_time_avg'] = 1.0
    df['vol_vs_hour_avg'] = 1.0
    
    # ============================================================
    # 9️⃣ VOLUME EXTREMES
    # ============================================================
    
    df['very_high_vol_30m'] = (df['vol_pctrank_6'] >= 0.90).astype(int)
    df['very_high_vol_1h'] = (df['vol_pctrank_12'] >= 0.90).astype(int)
    df['very_high_vol_2h'] = (df['vol_pctrank_24'] >= 0.90).astype(int)
    
    df['high_vol_30m'] = (df['vol_pctrank_6'] >= 0.80).astype(int)
    df['high_vol_1h'] = (df['vol_pctrank_12'] >= 0.80).astype(int)
    df['high_vol_2h'] = (df['vol_pctrank_24'] >= 0.80).astype(int)
    
    df['very_low_vol_30m'] = (df['vol_pctrank_6'] <= 0.10).astype(int)
    df['very_low_vol_1h'] = (df['vol_pctrank_12'] <= 0.10).astype(int)
    df['very_low_vol_2h'] = (df['vol_pctrank_24'] <= 0.10).astype(int)
    
    df['low_vol_30m'] = (df['vol_pctrank_6'] <= 0.20).astype(int)
    df['low_vol_1h'] = (df['vol_pctrank_12'] <= 0.20).astype(int)
    df['low_vol_2h'] = (df['vol_pctrank_24'] <= 0.20).astype(int)
    
    df['vol_spike_1.5x'] = (df['rvol_30m'] > 1.5).astype(int)
    df['vol_spike_2x'] = (df['rvol_30m'] > 2.0).astype(int)
    df['vol_spike_3x'] = (df['rvol_30m'] > 3.0).astype(int)
    
    # ============================================================
    # 10 VOLUME VOLATILITY
    # ============================================================
    
    vol_std_6 = df['volume'].rolling(6).std()
    vol_std_12 = df['volume'].rolling(12).std()
    vol_std_24 = df['volume'].rolling(24).std()
    
    df['vol_cv_30m'] = vol_std_6 / (vma_6 + 1e-8)
    df['vol_cv_1h'] = vol_std_12 / (vma_12 + 1e-8)
    df['vol_cv_2h'] = vol_std_24 / (vma_24 + 1e-8)
    
    df['vol_zscore_30m'] = (df['volume'] - vma_6) / (vol_std_6 + 1e-8)
    df['vol_zscore_1h'] = (df['volume'] - vma_12) / (vol_std_12 + 1e-8)
    df['vol_zscore_2h'] = (df['volume'] - vma_24) / (vol_std_24 + 1e-8)
    
    # ============================================================
    # 11 VOLUME CONSISTENCY
    # ============================================================
    
    high_vol_bars = (df['vol_pctrank_12'] > 0.7).astype(int)
    low_vol_bars = (df['vol_pctrank_12'] < 0.3).astype(int)
    
    consecutive_high_vol = []
    consecutive_low_vol = []
    high_count = 0
    low_count = 0
    
    for high, low in zip(high_vol_bars, low_vol_bars):
        if high:
            high_count += 1
            low_count = 0
        elif low:
            low_count += 1
            high_count = 0
        else:
            high_count = 0
            low_count = 0
        
        consecutive_high_vol.append(high_count)
        consecutive_low_vol.append(low_count)
    
    df['consecutive_high_vol'] = consecutive_high_vol
    df['consecutive_low_vol'] = consecutive_low_vol
    
    df['sustained_high_vol'] = (df['consecutive_high_vol'] >= 3).astype(int)
    df['sustained_low_vol'] = (df['consecutive_low_vol'] >= 3).astype(int)
    
    # ============================================================
    # 12 VOLUME-PRICE RELATIONSHIP
    # ============================================================
    
    df['vol_per_move'] = np.where(
        np.abs(df['ret_1']) > 0.0001,
        df['volume'] / np.abs(df['ret_1']),
        0
    )
    
    vol_per_move_avg = df['vol_per_move'].rolling(12).mean()
    df['vol_per_move_ratio'] = df['vol_per_move'] / (vol_per_move_avg + 1e-8)
    
    df['volume_on_up'] = np.where(df['ret_1'] > 0, df['volume'], 0)
    df['volume_on_down'] = np.where(df['ret_1'] < 0, df['volume'], 0)
    
    vol_up_sum = df['volume_on_up'].rolling(6).sum()
    vol_down_sum = df['volume_on_down'].rolling(6).sum()
    
    df['vol_up_down_ratio'] = np.where(
        vol_down_sum > 0,
        vol_up_sum / vol_down_sum,
        1.0
    )
    
    return df


# Apply to all symbols
for sym in dfs_by_symbol.keys():
    dfs_by_symbol[sym] = add_volume_features(dfs_by_symbol[sym].copy())

# Verification
for sym in dfs_by_symbol.keys():
    print(f"--- {sym.Value} ---")
    print(dfs_by_symbol[sym].head(10))
    print(f"\nColumns: {dfs_by_symbol[sym].columns.tolist()}")
    print()

### Volatility Features

In [12]:
def add_volatility_features(df):
    """
    Optimized stationary volatility features for 10-20 MINUTE holding periods (2-4 bars).
    
    OPTIMIZATION STRATEGY:
    - 70% of features focused on <1 hour horizons (most predictive)
    - 25% on 1-4 hour horizons (context)
    - 5% on >4 hour horizons (regime detection only)
    
    Focus: Realized volatility, range-based measures, volatility changes
    
    Time conversions for 5-min bars:
    - 1 hour = 12 bars
    - 4 hours = 48 bars
    - 1 day = 288 bars
    """
    df = df.copy()
    
    # Compute returns if not already present
    if 'ret_1' not in df.columns:
        df['ret_1'] = df['close'].pct_change(1)
    
    # ============================================================
    # 1️⃣ REALIZED VOLATILITY (Std Dev of Returns - Short Windows)
    # ============================================================
    
    # Very short-term realized volatility (critical!)
    df['rvol_3'] = df['ret_1'].rolling(3).std()    # 15-min realized vol
    df['rvol_6'] = df['ret_1'].rolling(6).std()    # 30-min realized vol - matches your hold
    df['rvol_12'] = df['ret_1'].rolling(12).std()  # 1-hour realized vol
    df['rvol_24'] = df['ret_1'].rolling(24).std()  # 2-hour realized vol
    df['rvol_48'] = df['ret_1'].rolling(48).std()  # 4-hour realized vol
    df['rvol_288'] = df['ret_1'].rolling(288).std() # 1-day realized vol (context)
    
    # ============================================================
    # 2️⃣ PARKINSON VOLATILITY (Range-Based - More Efficient)
    # ============================================================
    
    # Parkinson estimator using high-low range (better for short windows)
    # Formula: sqrt(1/(4*ln(2)) * mean((ln(high/low))^2))
    
    log_hl_sq = (np.log(df['high'] / df['low']) ** 2)
    
    df['park_vol_3'] = np.sqrt(log_hl_sq.rolling(3).mean() / (4 * np.log(2)))
    df['park_vol_6'] = np.sqrt(log_hl_sq.rolling(6).mean() / (4 * np.log(2)))
    df['park_vol_12'] = np.sqrt(log_hl_sq.rolling(12).mean() / (4 * np.log(2)))
    df['park_vol_24'] = np.sqrt(log_hl_sq.rolling(24).mean() / (4 * np.log(2)))
    
    # ============================================================
    # 3️⃣ GARMAN-KLASS VOLATILITY (Uses OHLC - Most Efficient)
    # ============================================================
    
    # Garman-Klass estimator (uses open, high, low, close)
    # More efficient than close-to-close, incorporates intrabar info
    
    log_hl = np.log(df['high'] / df['low'])
    log_co = np.log(df['close'] / df['open'])
    
    gk = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)
    
    df['gk_vol_3'] = np.sqrt(gk.rolling(3).mean())
    df['gk_vol_6'] = np.sqrt(gk.rolling(6).mean())
    df['gk_vol_12'] = np.sqrt(gk.rolling(12).mean())
    df['gk_vol_24'] = np.sqrt(gk.rolling(24).mean())
    
    # ============================================================
    # 4️⃣ AVERAGE TRUE RANGE (ATR) - Normalized
    # ============================================================
    
    # True Range components
    tr1 = df['high'] - df['low']
    tr2 = np.abs(df['high'] - df['close'].shift(1))
    tr3 = np.abs(df['low'] - df['close'].shift(1))
    
    true_range = np.maximum(tr1, np.maximum(tr2, tr3))
    
    # ATR at different horizons
    df['atr_3'] = true_range.rolling(3).mean()    # 15-min ATR
    df['atr_6'] = true_range.rolling(6).mean()    # 30-min ATR
    df['atr_12'] = true_range.rolling(12).mean()  # 1-hour ATR
    df['atr_24'] = true_range.rolling(24).mean()  # 2-hour ATR
    df['atr_48'] = true_range.rolling(48).mean()  # 4-hour ATR
    
    # Normalized ATR (as % of price)
    df['atr_3_pct'] = df['atr_3'] / df['close']
    df['atr_6_pct'] = df['atr_6'] / df['close']
    df['atr_12_pct'] = df['atr_12'] / df['close']
    df['atr_24_pct'] = df['atr_24'] / df['close']
    
    # ============================================================
    # 5️⃣ RELATIVE VOLATILITY (Vol vs Vol)
    # ============================================================
    
    # Short-term vol vs medium-term vol
    df['rvol_ratio_3_12'] = df['rvol_3'] / (df['rvol_12'] + 1e-8)   # 15m vs 1h
    df['rvol_ratio_6_24'] = df['rvol_6'] / (df['rvol_24'] + 1e-8)   # 30m vs 2h
    df['rvol_ratio_12_48'] = df['rvol_12'] / (df['rvol_48'] + 1e-8) # 1h vs 4h
    df['rvol_ratio_24_288'] = df['rvol_24'] / (df['rvol_288'] + 1e-8) # 2h vs 1d
    
    # ATR ratios (volatility acceleration)
    df['atr_ratio_3_12'] = df['atr_3'] / (df['atr_12'] + 1e-8)
    df['atr_ratio_6_24'] = df['atr_6'] / (df['atr_24'] + 1e-8)
    df['atr_ratio_12_48'] = df['atr_12'] / (df['atr_48'] + 1e-8)
    
    # ============================================================
    # 6️⃣ VOLATILITY PERCENTILE RANKS
    # ============================================================
    
    # Percentile rank of realized volatility
    df['rvol_pctrank_12'] = df['rvol_6'].rolling(12).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['rvol_pctrank_24'] = df['rvol_6'].rolling(24).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['rvol_pctrank_48'] = df['rvol_6'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # Percentile rank of ATR
    df['atr_pctrank_12'] = df['atr_6'].rolling(12).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['atr_pctrank_24'] = df['atr_6'].rolling(24).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    df['atr_pctrank_48'] = df['atr_6'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # 7️⃣ VOLATILITY REGIME (Binary Flags)
    # ============================================================
    
    # High volatility flags (top 20% and top 10%)
    df['high_vol_30m'] = (df['rvol_pctrank_12'] >= 0.80).astype(int)
    df['high_vol_1h'] = (df['rvol_pctrank_24'] >= 0.80).astype(int)
    df['high_vol_2h'] = (df['rvol_pctrank_48'] >= 0.80).astype(int)
    
    df['very_high_vol_30m'] = (df['rvol_pctrank_12'] >= 0.90).astype(int)
    df['very_high_vol_1h'] = (df['rvol_pctrank_24'] >= 0.90).astype(int)
    
    # Low volatility flags (bottom 20% and bottom 10%)
    df['low_vol_30m'] = (df['rvol_pctrank_12'] <= 0.20).astype(int)
    df['low_vol_1h'] = (df['rvol_pctrank_24'] <= 0.20).astype(int)
    df['low_vol_2h'] = (df['rvol_pctrank_48'] <= 0.20).astype(int)
    
    df['very_low_vol_30m'] = (df['rvol_pctrank_12'] <= 0.10).astype(int)
    df['very_low_vol_1h'] = (df['rvol_pctrank_24'] <= 0.10).astype(int)
    
    # Volatility spike (sudden increase)
    df['vol_spike'] = (df['rvol_ratio_3_12'] > 1.5).astype(int)
    df['vol_compression'] = (df['rvol_ratio_3_12'] < 0.7).astype(int)
    
    # ============================================================
    # 8️⃣ INTRABAR VOLATILITY (Within-Bar Measures)
    # ============================================================
    
    # High-low range as % of close (already have hl_range from price features)
    df['hl_range_vol'] = (df['high'] - df['low']) / df['close']
    
    # Rolling average of hl_range
    df['avg_hl_range_6'] = df['hl_range_vol'].rolling(6).mean()
    df['avg_hl_range_12'] = df['hl_range_vol'].rolling(12).mean()
    df['avg_hl_range_24'] = df['hl_range_vol'].rolling(24).mean()
    
    # Current range vs average range (range expansion/contraction)
    df['range_ratio_6'] = df['hl_range_vol'] / (df['avg_hl_range_6'] + 1e-8)
    df['range_ratio_12'] = df['hl_range_vol'] / (df['avg_hl_range_12'] + 1e-8)
    
    # Range expansion flags
    df['range_expansion'] = (df['range_ratio_6'] > 1.5).astype(int)
    df['range_contraction'] = (df['range_ratio_6'] < 0.7).astype(int)
    
    # ============================================================
    # 9️⃣ VOLATILITY MOMENTUM (Rate of Change)
    # ============================================================
    
    # Change in volatility (vol-of-vol)
    df['rvol_roc_3'] = df['rvol_6'].pct_change(3)   # 15-min vol change
    df['rvol_roc_6'] = df['rvol_6'].pct_change(6)   # 30-min vol change
    df['rvol_roc_12'] = df['rvol_12'].pct_change(12) # 1-hour vol change
    
    df['atr_roc_3'] = df['atr_6'].pct_change(3)
    df['atr_roc_6'] = df['atr_6'].pct_change(6)
    
    # Volatility acceleration (2nd derivative)
    df['rvol_accel'] = df['rvol_roc_3'] - df['rvol_roc_3'].shift(1)
    
    # Volatility trend flags
    df['vol_increasing'] = (df['rvol_roc_6'] > 0).astype(int)
    df['vol_decreasing'] = (df['rvol_roc_6'] < 0).astype(int)
    
    # ============================================================
    # 10 VOLATILITY CONSISTENCY (Streak Detection)
    # ============================================================
    
    # Consecutive high/low volatility bars
    high_vol_bars = (df['rvol_pctrank_24'] > 0.7).astype(int)
    low_vol_bars = (df['rvol_pctrank_24'] < 0.3).astype(int)
    
    consecutive_high_vol = []
    consecutive_low_vol = []
    high_count = 0
    low_count = 0
    
    for high, low in zip(high_vol_bars, low_vol_bars):
        if high:
            high_count += 1
            low_count = 0
        elif low:
            low_count += 1
            high_count = 0
        else:
            high_count = 0
            low_count = 0
        
        consecutive_high_vol.append(high_count)
        consecutive_low_vol.append(low_count)
    
    df['consecutive_high_vol'] = consecutive_high_vol
    df['consecutive_low_vol'] = consecutive_low_vol
    
    # Sustained volatility flags
    df['sustained_high_vol'] = (df['consecutive_high_vol'] >= 3).astype(int)
    df['sustained_low_vol'] = (df['consecutive_low_vol'] >= 3).astype(int)
    
    # ============================================================
    # 11 BOLLINGER BAND WIDTH (Volatility Proxy)
    # ============================================================
    
    # BB width at different horizons (stationary when normalized)
    sma_12 = df['close'].rolling(12).mean()
    sma_24 = df['close'].rolling(24).mean()
    sma_48 = df['close'].rolling(48).mean()
    
    std_12 = df['close'].rolling(12).std()
    std_24 = df['close'].rolling(24).std()
    std_48 = df['close'].rolling(48).std()
    
    # BB width as % of price
    df['bb_width_1h'] = (2 * std_12) / (sma_12 + 1e-8)
    df['bb_width_2h'] = (2 * std_24) / (sma_24 + 1e-8)
    df['bb_width_4h'] = (2 * std_48) / (sma_48 + 1e-8)
    
    # BB width percentile rank
    df['bb_width_pctrank'] = df['bb_width_1h'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # 12 VOLATILITY Z-SCORES
    # ============================================================
    
    # Z-score of volatility (volatility relative to its own distribution)
    rvol_6_mean = df['rvol_6'].rolling(48).mean()
    rvol_6_std = df['rvol_6'].rolling(48).std()
    df['rvol_zscore'] = (df['rvol_6'] - rvol_6_mean) / (rvol_6_std + 1e-8)
    
    atr_6_mean = df['atr_6'].rolling(48).mean()
    atr_6_std = df['atr_6'].rolling(48).std()
    df['atr_zscore'] = (df['atr_6'] - atr_6_mean) / (atr_6_std + 1e-8)
    
    return df


# Apply to all symbols
for sym in dfs_by_symbol.keys():
    dfs_by_symbol[sym] = add_volatility_features(dfs_by_symbol[sym].copy())

# Verification
for sym in dfs_by_symbol.keys():
    print(f"--- {sym.Value} ---")
    print(dfs_by_symbol[sym].head(10))
    print(f"\nVolatility columns: {[col for col in dfs_by_symbol[sym].columns if 'vol' in col.lower() or 'atr' in col.lower()]}")
    print()

### Merge Features

In [13]:
# Check if 'open' exists in BTC dataframe BEFORE merging
btc_symbol = [sym for sym in dfs_by_symbol.keys() if 'BTC' in sym.Value][0]

print("Columns in BTC dataframe BEFORE merge:")
print(dfs_by_symbol[btc_symbol].columns.tolist())

# Check if 'open' is there
if 'close' in dfs_by_symbol[btc_symbol].columns:
    print("\n✓ 'open' EXISTS in original BTC dataframe")
else:
    print("\n✗ 'open' DOES NOT EXIST in original BTC dataframe")

In [14]:
# Super simple inspection - no copies, no merges
print("Number of symbols in dfs_by_symbol:", len(dfs_by_symbol))
print("Keys in dfs_by_symbol:", list(dfs_by_symbol.keys()))

# Get first (and only) key
first_key = list(dfs_by_symbol.keys())[0]
print("\nFirst key:", first_key)

# Look at shape without copying
print("Shape:", dfs_by_symbol[first_key].shape)

# Look at columns without copying  
print("\nFirst 10 columns:")
print(dfs_by_symbol[first_key].columns[:10].tolist())

# Check memory
print(f"\nMemory: {dfs_by_symbol[first_key].memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [15]:
# Get your BTC data without copying
btc_symbol = list(dfs_by_symbol.keys())[0]
merged_df = dfs_by_symbol[btc_symbol]

# Rename columns in place (no copy)
merged_df.columns = ['BTCUSDT_' + col for col in merged_df.columns]

In [16]:
print(merged_df.head())

In [17]:
# Check if BTCUSDT_close exists AFTER merge
if 'BTCUSDT_close' in merged_df.columns:
    print("✓ BTCUSDT_close EXISTS after merge")
else:
    print("✗ BTCUSDT_close DOES NOT EXIST after merge")

# Show all BTCUSDT columns after merge
btc_cols = [col for col in merged_df.columns if 'BTCUSDT' in col]
print(f"\nAll BTCUSDT columns after merge ({len(btc_cols)} total):")
print(btc_cols)

### Assess Nans

In [18]:
def compute_warmup_period(merged_df):
    """
    Computes the warmup period - number of initial rows before all columns are valid.
    
    Args:
        merged_df: Merged DataFrame with all features
        
    Returns:
        warmup_period: Number of rows to skip from the beginning
    """
    
    # Find the first row where ALL columns have non-NaN values
    all_valid_mask = merged_df.notna().all(axis=1)
    
    if all_valid_mask.any():
        first_valid_idx = all_valid_mask.idxmax()
        warmup_period = merged_df.index.get_loc(first_valid_idx)
    else:
        warmup_period = len(merged_df)
    
    print(f"Warmup period: {warmup_period} rows")
    print(f"Total rows: {len(merged_df)}")
    print(f"Usable rows: {len(merged_df) - warmup_period}")
    
    return warmup_period


# Usage
warmup_period = compute_warmup_period(merged_df)

In [19]:
# If you don't need the original merged_df anymore
merged_df = merged_df.iloc[warmup_period:]

print(f"\ndf_clean shape: {merged_df.shape}")
print(f"Any NaN values remaining: {merged_df.isna().any().any()}")

In [20]:
def decide_on_nans_for_btc_prediction(df_clean):
    """
    Analyzes NaNs across entire dataset to decide on best action for BTC prediction.
    """
    
    print("\nNaN Analysis - Full Dataset:")
    print("=" * 60)
    
    total_rows = len(df_clean)
    total_cols = len(df_clean.columns)
    
    # Row-level analysis
    rows_with_nans = df_clean.isna().any(axis=1).sum()
    rows_clean = total_rows - rows_with_nans
    pct_rows_with_nans = (rows_with_nans / total_rows) * 100
    
    print(f"Total rows: {total_rows}")
    print(f"Rows with NaNs: {rows_with_nans} ({pct_rows_with_nans:.2f}%)")
    print(f"Clean rows: {rows_clean} ({100-pct_rows_with_nans:.2f}%)")
    
    # Column-level analysis
    print(f"\nTotal columns: {total_cols}")
    nan_counts = df_clean.isna().sum()
    cols_with_nans = nan_counts[nan_counts > 0]
    
    print(f"Columns with NaNs: {len(cols_with_nans)}")
    print("\nNaN % by column:")
    for col in cols_with_nans.sort_values(ascending=False).index:
        pct = (nan_counts[col] / total_rows) * 100
        print(f"  {col}: {pct:.2f}%")
    
    # Decision logic
    print("\n" + "=" * 60)
    print("DECISION:")
    print("=" * 60)
    
    if pct_rows_with_nans < 15:
        print(f"→ FORWARD FILL recommended")
        print(f"\nReason: {pct_rows_with_nans:.2f}% of rows have NaNs")
        print(f"For time series prediction, preserving continuity > losing data")
        action = "ffill"
    else:
        print(f"→ INVESTIGATE data quality issues")
        print(f"\nReason: {pct_rows_with_nans:.2f}% of rows have NaNs")
        print(f"High % suggests potential data collection problems")
        action = "investigate"
    
    return action


# Usage
action = decide_on_nans_for_btc_prediction(merged_df)

In [21]:
# Apply forward fill
print("\nApplying forward fill...")
print("=" * 60)

df_clean = merged_df
# Count NaNs before
nans_before = df_clean.isna().sum().sum()
rows_with_nans_before = df_clean.isna().any(axis=1).sum()

print(f"Before ffill:")
print(f"  Total NaNs: {nans_before}")
print(f"  Rows with NaNs: {rows_with_nans_before}")

# Apply forward fill
df_final = df_clean.ffill()

# Count NaNs after
nans_after = df_final.isna().sum().sum()
rows_with_nans_after = df_final.isna().any(axis=1).sum()

print(f"\nAfter ffill:")
print(f"  Total NaNs: {nans_after}")
print(f"  Rows with NaNs: {rows_with_nans_after}")

# Check if any NaNs remain
if nans_after > 0:
    print(f"\n⚠ WARNING: {nans_after} NaNs still remain!")
    remaining_nan_cols = df_final.isna().sum()
    remaining_nan_cols = remaining_nan_cols[remaining_nan_cols > 0]
    print("\nColumns with remaining NaNs:")
    for col, count in remaining_nan_cols.items():
        print(f"  {col}: {count}")
else:
    print("\n✓ All NaNs successfully filled!")

print(f"\nFinal dataset shape: {df_final.shape}")

In [22]:
print(df_final)

In [23]:
import pyarrow.parquet as pq
import pyarrow as pa

# Simplest version - just save with max compression
pq.write_table(
    pa.Table.from_pandas(df_final),
    'output.parquet',
    compression='zstd',
    compression_level=9
)